# Аналитика отключений систем ЖКХ во Владивостоке

## Загрузка библиотек

In [1]:
import numpy as np
import pandas as pd
import sqlite3

import re

from IPython.display import display, HTML

from typing import Any, Tuple

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer

import pymorphy3
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE

import plotly.express as px

import pickle

import warnings
from tqdm import tqdm

tqdm.pandas()
warnings.filterwarnings('ignore')

## Работа с данными

### Описание таблиц и столбцов

**blackouts** — таблица отключений систем ЖКХ 
- `id` — уникальный идентификатор  
- `start_date` — время начала отключения  
- `end_date` — время окончания отключения  
- `description` — описание отключения  
- `type` — тип отключения  
- `initiator_name` — инициатор отключения  
- `source` — источник информации  

**blackouts_buildings** — связь отключений со зданиями (многие-ко-многим)  
- `blackout_id` — ссылка на отключение  
- `building_id` — ссылка на здание  

**buildings** — справочник зданий  
- `id` — уникальный идентификатор  
- `street_id` — ссылка на улицу  
- `number` — номер дома  
- `district_id` — ссылка на официальный район  
- `is_fake` — признак фейкового здания  
- `folk_district_id` — ссылка на народный район  
- `big_folk_district_id` — ссылка на большой народный район  
- `type` — тип здания  
- `city_id` — ссылка на город  
- `coordinates` — географические координаты  

**cities** — справочник городов  
- `id` — уникальный идентификатор  
- `name` — название города  

**districts** — справочник официальных административных районов 
- `id` — уникальный идентификатор  
- `name` — название района  

**folk_districts** — справочник народных/неформальных районов
- `id` — уникальный идентификатор  
- `name` — название района  

**big_folk_districts** — справочник крупных народных районов  
- `id` — уникальный идентификатор  
- `name` — название района  

**streets** — справочник улиц  
- `id` — уникальный идентификатор  
- `name` — название улицы  
- `city_id` — ссылка на город

### Загрузка данных из базы данных

In [2]:
DB_PATH = "Кейс_Аналитика.db"

db_connect = sqlite3.connect(DB_PATH)

In [3]:
blackouts = pd.read_sql("SELECT * FROM blackouts", con=db_connect)

blackouts.head()

,id,start_date,end_date,description,type,initiator_name,source
0,f88cefa506f44ebf8f010b8681b5449e,2018-01-01 00:08:00,2018-01-01 09:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...
1,38ddf6852801fa90cc70f9770239961e,2018-01-01 00:24:00,2018-01-01 10:00:00,"Авария на электролинии, остановка работы насос...",cold_water,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...
2,53c570099fe380dce9e56e2ace9cfa9c,2018-01-01 10:44:00,2018-01-02 18:00:00,Авария в системе водоснабжения дома. Жителям н...,hot_water,"ООО ""Управляющая компания ""Регион-ЖКХ""","Аварийная служба ООО ""Мадикс"""
3,9c1b9ebbd9a698eef046b27cb3568745,2018-01-01 11:32:00,2018-01-01 15:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...
4,8aa631cb343aac0731bbde806dfa6d8c,2018-01-01 11:33:00,2018-01-01 15:00:00,"Авария на электролинии, остановка работы насос...",hot_water,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...


In [4]:
buildings = pd.read_sql("SELECT * FROM buildings", con=db_connect)

buildings.head()

,id,street_id,number,district_id,is_fake,folk_district_id,big_folk_district_id,type,city_id,coordinates
0,b428b92bb123994a56234bb6eeeed414,37454c5a86ea320f2d5eb4eabdf1ae86,1,504f6b8bd5daeba64beb62bda8b3c12a,0,551a73c4f1b8d1cc72eea5ec752463c4,9fec0dfb8ff6bb1cf6126dc933be3489,нежилое,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.14199293270051, ""lon"": 131.9080723..."
1,24aa25303f0f91ede8ee0c0d324c8ee3,37454c5a86ea320f2d5eb4eabdf1ae86,100,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.171246518452826, ""lon"": 131.917043..."
2,504f6b8bd5daeba64beb62bda8b3c12a,37454c5a86ea320f2d5eb4eabdf1ae86,100А,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17160112621735, ""lon"": 131.9178653..."
3,87d459096286c57ed5e780205f041e68,37454c5a86ea320f2d5eb4eabdf1ae86,100В,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17208966911542, ""lon"": 131.9183524..."
4,a4f6e19b37548845b4ee10c573809e68,37454c5a86ea320f2d5eb4eabdf1ae86,102,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.172186535721075, ""lon"": 131.917439..."


In [5]:
blackouts_buildings = pd.read_sql("SELECT * FROM blackouts_buildings", con=db_connect)

blackouts_buildings.head()

,blackout_id,building_id
0,None,625c9861f138e4dfeb68cc9b1737d525
1,None,2ef83bdea9aed0734f44337c3ecf9a9a
2,None,cbfc294aceb59a85354aa0648e7e0e47
3,None,98462c9c6a1a96cec4f1e4b8e5c374fa
4,None,31bf19dff7df028f177ac45444b500b8


In [6]:
cities = pd.read_sql("SELECT * FROM cities", con=db_connect)

cities.head()

,id,name
0,24aa25303f0f91ede8ee0c0d324c8ee3,Артем
1,b428b92bb123994a56234bb6eeeed414,Владивосток


In [7]:
districts = pd.read_sql("SELECT * FROM districts", con=db_connect)

districts.head()

,id,name
0,b428b92bb123994a56234bb6eeeed414,Ленинский район
1,24aa25303f0f91ede8ee0c0d324c8ee3,Первомайский район
2,504f6b8bd5daeba64beb62bda8b3c12a,Первореченский район
3,87d459096286c57ed5e780205f041e68,Советский район
4,a4f6e19b37548845b4ee10c573809e68,Фрунзенский район


In [8]:
folk_districts = pd.read_sql("SELECT * FROM folk_districts", con=db_connect)

folk_districts.head()

,id,name
0,b428b92bb123994a56234bb6eeeed414,Центр
1,24aa25303f0f91ede8ee0c0d324c8ee3,БАМ
2,504f6b8bd5daeba64beb62bda8b3c12a,Гризодубова-Сафонова
3,87d459096286c57ed5e780205f041e68,Щитовая
4,a4f6e19b37548845b4ee10c573809e68,Трасса Де-Фриз - Седанка


In [9]:
big_folk_districts = pd.read_sql("SELECT * FROM big_folk_districts", con=db_connect)

big_folk_districts.head()

,id,name
0,b428b92bb123994a56234bb6eeeed414,Центр
1,24aa25303f0f91ede8ee0c0d324c8ee3,БАМ
2,504f6b8bd5daeba64beb62bda8b3c12a,Тихая
3,87d459096286c57ed5e780205f041e68,Шамора
4,a4f6e19b37548845b4ee10c573809e68,Пригород


In [10]:
streets = pd.read_sql("SELECT * FROM streets", con=db_connect)

streets.head()

,id,name,city_id
0,ceba6f5d616341f44f5fcae0425d020b,1-й Байкальский пер.,24aa25303f0f91ede8ee0c0d324c8ee3
1,a7fc0a136f4b1e18a8365d3d29c65c05,1-й Вокзальный пер.,24aa25303f0f91ede8ee0c0d324c8ee3
2,d3ef58b013fa6d05ad5b6cf4210e0fc7,1-й Воровского пер.,24aa25303f0f91ede8ee0c0d324c8ee3
3,4a60bf748c91c074046b86bd48bfa10a,1-й Зареченский пер.,24aa25303f0f91ede8ee0c0d324c8ee3
4,11edbe94c0140fae83c25cd72d11aa0b,1-й Линейный пер.,24aa25303f0f91ede8ee0c0d324c8ee3


### Разведочный анализ данных (EDA)

#### Кол-во строк и типы данных колонок

##### Таблица отключений систем ЖКХ

In [11]:
blackouts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25264 entries, 0 to 25263
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              25264 non-null  object
 1   start_date      25264 non-null  object
 2   end_date        25264 non-null  object
 3   description     25264 non-null  object
 4   type            25264 non-null  object
 5   initiator_name  25264 non-null  object
 6   source          8722 non-null   object
dtypes: object(7)
memory usage: 1.3+ MB


In [11]:
blackouts["start_date"] = pd.to_datetime(blackouts["start_date"])
blackouts["end_date"] = pd.to_datetime(blackouts["end_date"])

Колонки с датами были приведены к типу **datetime**.

##### Справочник зданий

In [13]:
buildings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58199 entries, 0 to 58198
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    58199 non-null  object
 1   street_id             58199 non-null  object
 2   number                58199 non-null  object
 3   district_id           34606 non-null  object
 4   is_fake               58199 non-null  int64 
 5   folk_district_id      26009 non-null  object
 6   big_folk_district_id  33063 non-null  object
 7   type                  34631 non-null  object
 8   city_id               58199 non-null  object
 9   coordinates           50521 non-null  object
dtypes: int64(1), object(9)
memory usage: 4.4+ MB


##### Связь отключений со зданиями

In [14]:
blackouts_buildings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1419556 entries, 0 to 1419555
Data columns (total 2 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   blackout_id  175590 non-null   object
 1   building_id  1419556 non-null  object
dtypes: object(2)
memory usage: 21.7+ MB


##### Справочник городов

In [15]:
cities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2 non-null      object
 1   name    2 non-null      object
dtypes: object(2)
memory usage: 160.0+ bytes


##### Справочник официальных административных районов

In [16]:
districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      5 non-null      object
 1   name    5 non-null      object
dtypes: object(2)
memory usage: 208.0+ bytes


##### Справочник народных/неформальных районов

In [17]:
folk_districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3594 entries, 0 to 3593
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      3594 non-null   object
 1   name    3594 non-null   object
dtypes: object(2)
memory usage: 56.3+ KB


##### Справочник крупных народных районов

In [18]:
big_folk_districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      45 non-null     object
 1   name    45 non-null     object
dtypes: object(2)
memory usage: 848.0+ bytes


##### Справочник улиц

In [19]:
streets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2023 entries, 0 to 2022
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       2023 non-null   object
 1   name     2023 non-null   object
 2   city_id  2023 non-null   object
dtypes: object(3)
memory usage: 47.5+ KB


#### Очистка пустых значений

In [12]:
def print_null_pct(data: pd.DataFrame):
    for col in data.columns:
        null_pct = (data[col].isnull().sum() / len(data) * 100).round(2)
        print(f"{col}: {null_pct}%")

##### Таблица отключений систем ЖКХ

In [21]:
print_null_pct(blackouts)

id: 0.0%
start_date: 0.0%
end_date: 0.0%
description: 0.0%
type: 0.0%
initiator_name: 0.0%
source: 65.48%


In [13]:
blackouts["source"] = blackouts["source"].fillna("Неизвестно")

В таблице с отключениями ЖКХ в столбце **«источник информации»** обнаружено 65% пропусков. Восстановить эти данные не представляется возможным из-за отсутствия точных исходных сведений. Для сохранения целостности данных все пропуски будут заполнены значением **«Неизвестно»**.

##### Справочник зданий

In [23]:
print_null_pct(buildings)

id: 0.0%
street_id: 0.0%
number: 0.0%
district_id: 40.54%
is_fake: 0.0%
folk_district_id: 55.31%
big_folk_district_id: 43.19%
type: 40.5%
city_id: 0.0%
coordinates: 13.19%


In [14]:
real_buildings = buildings[buildings["is_fake"] == 0]

print_null_pct(real_buildings)

id: 0.0%
street_id: 0.0%
number: 0.0%
district_id: 40.85%
is_fake: 0.0%
folk_district_id: 53.5%
big_folk_district_id: 40.95%
type: 35.3%
city_id: 0.0%
coordinates: 0.0%


Пропуски в координатах присутствуют только в строках со зданиями, которые считаются фейками.

In [15]:
tmp_cities = cities.copy().rename(columns={
    "id": "city_id",
    "name": "city_name"
})

tmp_districts = districts.copy().rename(columns={
    "id": "district_id",
    "name": "district_name"
})

tmp_folk_districts = folk_districts.copy().rename(columns={
    "id": "folk_district_id",
    "name": "folk_district_name"
})

tmp_big_folk_districts = big_folk_districts.copy().rename(columns={
    "id": "big_folk_district_id",
    "name": "big_folk_district_name"
})

real_buildings = real_buildings.merge(tmp_cities, on="city_id", how="left")
real_buildings = real_buildings.merge(tmp_districts, on="district_id", how="left")
real_buildings = real_buildings.merge(tmp_folk_districts, on="folk_district_id", how="left")
real_buildings = real_buildings.merge(tmp_big_folk_districts, on="big_folk_district_id", how="left")

In [16]:
real_buildings[["district_name", "folk_district_name", "big_folk_district_name"]] = real_buildings[
    ["district_name", "folk_district_name", "big_folk_district_name"]
].fillna("Неизвестно")

In [27]:
display(HTML(real_buildings.groupby(
    ["city_name", "district_name", "big_folk_district_name", "folk_district_name"]
)["id"].count().reset_index().rename(columns={"id": "count"}).to_html()))

,city_name,district_name,big_folk_district_name,folk_district_name,count
0,Артем,Неизвестно,Неизвестно,Неизвестно,16092
1,Артем,Неизвестно,Неизвестно,Район Хребта Богатая Грива,10
2,Артем,Неизвестно,Неизвестно,Тавайза,1
3,Артем,Неизвестно,Неизвестно,Трудовое,298
4,Артем,Неизвестно,Неизвестно,Угловое,2421
5,Владивосток,Ленинский район,"64, 71 микрорайоны",64 71 микрорайон,2
6,Владивосток,Ленинский район,"64, 71 микрорайоны","64, 71 микрорайон",346
7,Владивосток,Ленинский район,"64, 71 микрорайоны",Неизвестно,181
8,Владивосток,Ленинский район,"64, 71 микрорайоны",ТЭЦ(Золоотвал),1
9,Владивосток,Ленинский район,"64, 71 микрорайоны",Фадеева,16


Для восстановления пропущенных значений районов у зданий наиболее точным решением является обучение ML-модели, которая будет предсказывать район по географическим координатам. Этот подход использует существующие данные без привлечения внешних источников. Но реализовывать это будем потом.

##### Связь отключений со зданиями

In [28]:
print_null_pct(blackouts_buildings)

blackout_id: 87.63%
building_id: 0.0%


In [16]:
blackouts_buildings = blackouts_buildings.dropna()

##### Справочник городов

In [30]:
print_null_pct(cities)

id: 0.0%
name: 0.0%


##### Справочник официальных административных районов

In [31]:
print_null_pct(districts)

id: 0.0%
name: 0.0%


##### Справочник народных/неформальных районов

In [32]:
print_null_pct(folk_districts)

id: 0.0%
name: 0.0%


##### Справочник крупных народных районов

In [33]:
print_null_pct(big_folk_districts)

id: 0.0%
name: 0.0%


##### Справочник улиц

In [34]:
print_null_pct(streets)

id: 0.0%
name: 0.0%
city_id: 0.0%


### Предобработка данных (преобразование существующих, добавление новых)

#### Разбиение колонки координат на отдельные колонки (широта, долгота)

In [17]:
def separate_coordinates(coordinates: Any) -> pd.Series:
    if coordinates:
        regex = "-*\d+.\d+"
        separated_coordinates = re.findall(regex, coordinates)

        if len(separated_coordinates) != 2:
            return pd.Series([None, None])
        return pd.Series(separated_coordinates).astype(np.float64)
    
    return pd.Series([None, None])

In [18]:
buildings[["latitude", "longitude"]] = buildings["coordinates"].progress_apply(separate_coordinates)

100%|██████████| 58199/58199 [00:05<00:00, 11103.95it/s]


In [37]:
buildings.head()

,id,street_id,number,district_id,is_fake,folk_district_id,big_folk_district_id,type,city_id,coordinates,latitude,longitude
0,b428b92bb123994a56234bb6eeeed414,37454c5a86ea320f2d5eb4eabdf1ae86,1,504f6b8bd5daeba64beb62bda8b3c12a,0,551a73c4f1b8d1cc72eea5ec752463c4,9fec0dfb8ff6bb1cf6126dc933be3489,нежилое,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.14199293270051, ""lon"": 131.9080723...",43.141993,131.908072
1,24aa25303f0f91ede8ee0c0d324c8ee3,37454c5a86ea320f2d5eb4eabdf1ae86,100,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.171246518452826, ""lon"": 131.917043...",43.171247,131.917043
2,504f6b8bd5daeba64beb62bda8b3c12a,37454c5a86ea320f2d5eb4eabdf1ae86,100А,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17160112621735, ""lon"": 131.9178653...",43.171601,131.917865
3,87d459096286c57ed5e780205f041e68,37454c5a86ea320f2d5eb4eabdf1ae86,100В,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17208966911542, ""lon"": 131.9183524...",43.172090,131.918352
4,a4f6e19b37548845b4ee10c573809e68,37454c5a86ea320f2d5eb4eabdf1ae86,102,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.172186535721075, ""lon"": 131.917439...",43.172187,131.917440


#### Предсказание района по координатам

In [19]:
buildings_with_coords = buildings[buildings["latitude"].notna()]

In [20]:
buildings_for_model = buildings_with_coords[buildings_with_coords[[
    "district_id", "folk_district_id", "big_folk_district_id"
]].notna().all(axis=1)]

Создан датасет, на котором будет проходить обучение модели для предсказания района по координатам

In [25]:
buildings_for_model["target_district"] = buildings_for_model["district_id"] + "_" + \
      buildings_for_model["folk_district_id"] + "_" + buildings_for_model["big_folk_district_id"]

In [26]:
target_districts = buildings_for_model.groupby("target_district")[
    ["id"]].count().rename(columns={"id": "count"})
target_districts = target_districts[target_districts["count"] > 1].index # Отбираем целевые признаки, где хотя бы 2 примера

buildings_for_model = buildings_for_model[buildings_for_model["target_district"].isin(target_districts)]

Мы формируем целевую переменную как комбинацию трех идентификаторов районов в формате **[district_id]\_[folk_district_id]\_[big_folk_district_id]**. Модель, обученная на координатах, будет предсказывать эту комбинированную метку для зданий с пропущенными районами, после чего результат разделяется на три исходных признака.

In [27]:
X = buildings_for_model[["latitude", "longitude"]]
y = buildings_for_model["target_district"]

district_encoder = LabelEncoder()
y = district_encoder.fit_transform(y)

In [47]:
models_param_grid = [
    {
        'name': 'RandomForest',
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    {
        'name': 'XGBoost',
        'model': XGBClassifier(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 6, 9],
            'colsample_bytree': [0.8, 0.9, 1.0]
        }
    },
    {
        'name': 'LightGBM', 
        'model': LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [5, 10, 15]
        }
    },
    {
        'name': 'GradientBoosting',
        'model': GradientBoostingClassifier(random_state=42),
        'param_grid': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7]
        }
    }
]

In [48]:
f1_weighted_scorer = make_scorer(f1_score, average='weighted')

best_score = 0
district_predictor = None
district_predictor_name = ""

stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Сравнение моделей ---")

for model_config in models_param_grid:
    print(f"Обучение {model_config['name']}")

    try:
        grid_search = GridSearchCV(
            estimator=model_config['model'],
            param_grid=model_config['param_grid'],
            scoring=f1_weighted_scorer,
            cv=stratified_cv,
            n_jobs=-1,
            verbose=1,
            return_train_score=True
        )
        
        grid_search.fit(X, y)
        
        print(f"Лучший F1 Weighted Score для {model_config['name']}: {grid_search.best_score_:.4f}")
        print(f"Лучшие параметры: {grid_search.best_params_}")
        
        if grid_search.best_score_ > best_score:
            best_score = grid_search.best_score_
            district_predictor = grid_search.best_estimator_
            district_predictor_name = model_config['name']
    
    except Exception as e:
        print(f"Ошибка при обучении {model_config['name']}: {e}")

print(f"Лучшая модель: {district_predictor_name}")
print(f"Лучший F1 Weighted Score: {best_score:.4f}")

--- Сравнение моделей ---
Обучение RandomForest
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Лучший F1 Weighted Score для RandomForest: 0.9704
Лучшие параметры: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Обучение XGBoost
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Лучший F1 Weighted Score для XGBoost: 0.9561
Лучшие параметры: {'colsample_bytree': 1.0, 'max_depth': 6, 'n_estimators': 200}
Обучение LightGBM
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Лучший F1 Weighted Score для LightGBM: 0.0696
Лучшие параметры: {'max_depth': 15, 'n_estimators': 100}
Обучение GradientBoosting
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Лучший F1 Weighted Score для GradientBoosting: 0.9443
Лучшие параметры: {'max_depth': 7, 'n_estimators': 200}
Лучшая модель: RandomForest
Лучший F1 Weighted Score: 0.9704


In [51]:
cv_scores_weighted = []
cv_scores_macro = []

for fold, (train_idx, val_idx) in enumerate(stratified_cv.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    district_predictor.fit(X_train, y_train)
    y_pred = district_predictor.predict(X_val)
    
    f1_weighted = f1_score(y_val, y_pred, average='weighted')
    f1_macro = f1_score(y_val, y_pred, average='macro')
    cv_scores_weighted.append(f1_weighted)
    cv_scores_macro.append(f1_macro)
    print(f"Fold {fold}: F1 Weighted = {f1_weighted:.4f}, F1 Macro = {f1_macro:.4f}")

print(f"Средний F1 Weighted: {np.mean(cv_scores_weighted):.4f} (+/- {np.std(cv_scores_weighted):.4f})")
print(f"Средний F1 Macro: {np.mean(cv_scores_macro):.4f} (+/- {np.std(cv_scores_macro):.4f})")

Fold 1: F1 Weighted = 0.9692, F1 Macro = 0.8183
Fold 2: F1 Weighted = 0.9733, F1 Macro = 0.8499
Fold 3: F1 Weighted = 0.9667, F1 Macro = 0.7874
Fold 4: F1 Weighted = 0.9736, F1 Macro = 0.8443
Fold 5: F1 Weighted = 0.9690, F1 Macro = 0.8186
Средний F1 Weighted: 0.9704 (+/- 0.0027)
Средний F1 Macro: 0.8237 (+/- 0.0223)


Модель предсказания районов по координатам показывает высокую эффективность: F1 Weighted 0.9704 при стабильности across фолдов. Разрыв между F1 Weighted и F1 Macro (0.8237) указывает на дисбаланс классов - модель хуже предсказывает малые районы (так как представлено либо мало примеров, либо они очень близки к другим микрорайонам).

Решение готово к внедрению для автоматического заполнения пропусков в данных.

In [52]:
with open("District_Predictor.sav", "wb") as f:
    pickle.dump(district_predictor, f)

In [21]:
with open("District_Predictor.sav", "rb") as f:
    district_predictor = pickle.load(f)

In [28]:
def fill_district(row: pd.Series) -> pd.Series:
    district_id = row.district_id
    folk_district_id = row.folk_district_id
    big_folk_district_id = row.big_folk_district_id
    latitude = row.latitude
    longitude = row.longitude

    if district_id is None or folk_district_id is None or big_folk_district_id is None:
        if latitude and longitude:
            prediction = district_predictor.predict([[latitude, longitude]])
            prediction = district_encoder.inverse_transform(prediction)[0]
            district_id, folk_district_id, big_folk_district_id = prediction.split("_")
        else:
            pd.Series([None, None, None, None, None])
        
    return pd.Series([district_id, folk_district_id, big_folk_district_id, latitude, longitude])

In [29]:
district_cols = ["district_id", "folk_district_id", "big_folk_district_id", "latitude", "longitude"]

buildings[district_cols] = buildings[district_cols].progress_apply(fill_district, axis=1)

100%|██████████| 58199/58199 [20:00<00:00, 48.48it/s]  


#### Предобработка признака - тип постройки

In [56]:
buildings.groupby("type")["id"].count()

type
Жилое                          433
Жилое сельское                 576
Жилое сельское строение        291
Жилое строение                  87
Нежилое                        204
Нежилое строение                76
Постройка без адреса             3
Производственное                27
Производственное строение        1
Разрушенное                      3
СНТ                            156
Строение жилое                   4
Строение жилое сельское          3
Строение нежилое                 3
Строение производственное        1
Строение строящееся              2
Строящееся                      63
Строящееся строение             15
гаражи                          12
дача                           914
жилое многоквартирное         4836
нежилое                       5328
общественное                   337
планируемое                     39
производственное               290
разрушенное                     91
строение                        55
строящееся                     840
частный дом    

In [30]:
group_mapping = {
    # Жилые объекты
    "жилое": "Жилые объекты",
    "жилое строение": "Жилые объекты",
    "жилое многоквартирное": "Многоквартирные дома",
    "строение жилое": "Жилые объекты",

    # Сельские жилые
    "жилое сельское": "Сельские жилые объекты",
    "жилое сельское строение": "Сельские жилые объекты",
    "строение жилое сельское": "Сельские жилые объекты",

    # Частные дома, дачи, СНТ
    "частный дом": "Частные дома",
    "дача": "Дачи",
    "снт": "Садовые и дачные участки",

    # Нежилые
    "нежилое": "Нежилые объекты",
    "нежилое строение": "Нежилые объекты",
    "строение нежилое": "Нежилые объекты",
    "общественное": "Общественные объекты",

    # Производственные
    "производственное": "Производственные объекты",
    "производственное строение": "Производственные объекты",
    "строение производственное": "Производственные объекты",

    # Строящиеся / планируемые
    "строящееся": "Строящиеся объекты",
    "строящееся строение": "Строящиеся объекты",
    "строение строящееся": "Строящиеся объекты",
    "планируемое": "Планируемые объекты",

    # Разрушенные / без адреса
    "разрушенное": "Разрушенные объекты",
    "постройка без адреса": "Безадресные постройки",

    # Прочие
    "строение": "Прочие постройки",
    "гаражи": "Гаражи"
}


In [31]:
def prepare_building_type(building_type: str):
    if building_type is None:
        return None
    else:
        building_type = building_type.lower()
        return group_mapping.get(building_type, None)

In [32]:
buildings["type"] = buildings["type"].progress_apply(prepare_building_type)

100%|██████████| 58199/58199 [00:00<00:00, 1352723.12it/s]


In [ ]:
buildings.groupby("type")["id"].count()

type
Безадресные постройки           3
Гаражи                         12
Дачи                          914
Жилые объекты                 524
Многоквартирные дома         4836
Нежилые объекты              5611
Общественные объекты          337
Планируемые объекты            39
Производственные объекты      319
Прочие постройки               55
Разрушенные объекты            94
Садовые и дачные участки      156
Сельские жилые объекты        870
Строящиеся объекты            920
Частные дома                19941
Name: id, dtype: int64

Типы зданий сгруппированы по общим характеристикам. Для восстановления пропущенных значений будет использована модель машинного обучения, обученная на существующих данных.

In [121]:
buildings_with_type = buildings[(buildings["type"].notna()) & (buildings["latitude"].notna())]

print_null_pct(buildings_with_type)

id: 0.0%
street_id: 0.0%
number: 0.0%
district_id: 0.0%
is_fake: 0.0%
folk_district_id: 0.0%
big_folk_district_id: 0.0%
type: 0.0%
city_id: 0.0%
coordinates: 0.0%
latitude: 0.0%
longitude: 0.0%


In [98]:
buildings_with_type["district_feature"] = buildings_with_type["district_id"] + "_" + \
      buildings_with_type["folk_district_id"] + "_" + buildings_with_type["big_folk_district_id"]

In [122]:
categorical_columns = ["street_id"]

district_feature_encoder = OneHotEncoder(sparse_output=False)

one_hot_encoded = district_feature_encoder.fit_transform(buildings_with_type[categorical_columns])

one_hot_df = pd.DataFrame(one_hot_encoded, 
                          columns=district_feature_encoder.get_feature_names_out(categorical_columns))

buildings_with_type = pd.concat([
    buildings_with_type.drop(categorical_columns, axis=1).reset_index(drop=True),
    one_hot_df.reset_index(drop=True)
], axis=1)

In [124]:
X = buildings_with_type[["latitude", "longitude"] + list(one_hot_df.columns.values)]
y = buildings_with_type["type"]

type_encoder = LabelEncoder()
y = type_encoder.fit_transform(y)

In [125]:
best_score = 0
building_type_predictor = None
building_type_predictor_name = ""

stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Сравнение моделей ---")

for model_config in models_param_grid:
    print(f"Обучение {model_config['name']}")
    
    grid_search = GridSearchCV(
        estimator=model_config['model'],
        param_grid=model_config['param_grid'],
        scoring=f1_weighted_scorer,
        cv=stratified_cv,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )
    
    grid_search.fit(X, y)
    
    print(f"Лучший F1 Weighted Score для {model_config['name']}: {grid_search.best_score_:.4f}")
    print(f"Лучшие параметры: {grid_search.best_params_}")
    
    if grid_search.best_score_ > best_score:
        best_score = grid_search.best_score_
        building_type_predictor = grid_search.best_estimator_
        building_type_predictor_name = model_config['name']

print(f"Лучшая модель: {building_type_predictor_name}")
print(f"Лучший F1 Weighted Score: {best_score:.4f}")

--- Сравнение моделей ---
Обучение RandomForest
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Лучший F1 Weighted Score для RandomForest: 0.8274
Лучшие параметры: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Обучение XGBoost
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Лучший F1 Weighted Score для XGBoost: 0.8031
Лучшие параметры: {'colsample_bytree': 1.0, 'max_depth': 9, 'n_estimators': 300}
Обучение LightGBM
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Лучший F1 Weighted Score для LightGBM: 0.6374
Лучшие параметры: {'max_depth': 5, 'n_estimators': 100}
Обучение GradientBoosting
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Лучший F1 Weighted Score для GradientBoosting: 0.7964
Лучшие параметры: {'max_depth': 7, 'n_estimators': 200}
Лучшая модель: RandomForest
Лучший F1 Weighted Score: 0.8274


In [126]:
with open("Building_Type_Predictor.sav", "wb") as f:
    pickle.dump(building_type_predictor, f)

In [33]:
with open("Building_Type_Predictor.sav", "rb") as f:
    building_type_predictor = pickle.load(f)

In [128]:
cv_scores_weighted = []
cv_scores_macro = []

for fold, (train_idx, val_idx) in enumerate(stratified_cv.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    building_type_predictor.fit(X_train, y_train)
    y_pred = building_type_predictor.predict(X_val)
    
    f1_weighted = f1_score(y_val, y_pred, average='weighted')
    f1_macro = f1_score(y_val, y_pred, average='macro')
    cv_scores_weighted.append(f1_weighted)
    cv_scores_macro.append(f1_macro)
    print(f"Fold {fold}: F1 Weighted = {f1_weighted:.4f}, F1 Macro = {f1_macro:.4f}")

print(f"Средний F1 Weighted: {np.mean(cv_scores_weighted):.4f} (+/- {np.std(cv_scores_weighted):.4f})")
print(f"Средний F1 Macro: {np.mean(cv_scores_macro):.4f} (+/- {np.std(cv_scores_macro):.4f})")

Fold 1: F1 Weighted = 0.8224, F1 Macro = 0.3570
Fold 2: F1 Weighted = 0.8330, F1 Macro = 0.4413
Fold 3: F1 Weighted = 0.8274, F1 Macro = 0.3760
Fold 4: F1 Weighted = 0.8265, F1 Macro = 0.3616
Fold 5: F1 Weighted = 0.8278, F1 Macro = 0.3852
Средний F1 Weighted: 0.8274 (+/- 0.0034)
Средний F1 Macro: 0.3842 (+/- 0.0303)


Модели машинного обучения, обученные на различных комбинациях географических и административных признаков, не смогли достичь приемлемой точности в определении типа здания. Восстановление данных через внешние источники также не дало результатов. Для обработки пропусков в колонке **type** будет использовано значение "Неизвестно".

In [34]:
buildings["type"] = buildings["type"].fillna("Неизвестно")

print_null_pct(buildings)

id: 0.0%
street_id: 0.0%
number: 0.0%
district_id: 0.0%
is_fake: 0.0%
folk_district_id: 0.0%
big_folk_district_id: 0.0%
type: 0.0%
city_id: 0.0%
coordinates: 13.19%
latitude: 13.24%
longitude: 13.24%


#### Создание признаков времени (месяц, продолжительность отключения в минутах)

In [132]:
blackouts.head()

,id,start_date,end_date,description,type,initiator_name,source
0,f88cefa506f44ebf8f010b8681b5449e,2018-01-01 00:08:00,2018-01-01 09:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...
1,38ddf6852801fa90cc70f9770239961e,2018-01-01 00:24:00,2018-01-01 10:00:00,"Авария на электролинии, остановка работы насос...",cold_water,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...
2,53c570099fe380dce9e56e2ace9cfa9c,2018-01-01 10:44:00,2018-01-02 18:00:00,Авария в системе водоснабжения дома. Жителям н...,hot_water,"ООО ""Управляющая компания ""Регион-ЖКХ""","Аварийная служба ООО ""Мадикс"""
3,9c1b9ebbd9a698eef046b27cb3568745,2018-01-01 11:32:00,2018-01-01 15:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...
4,8aa631cb343aac0731bbde806dfa6d8c,2018-01-01 11:33:00,2018-01-01 15:00:00,"Авария на электролинии, остановка работы насос...",hot_water,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...


In [35]:
blackouts["month"] = blackouts["start_date"].dt.month

blackouts.head()

,id,start_date,end_date,description,type,initiator_name,source,month
0,f88cefa506f44ebf8f010b8681b5449e,2018-01-01 00:08:00,2018-01-01 09:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...,1
1,38ddf6852801fa90cc70f9770239961e,2018-01-01 00:24:00,2018-01-01 10:00:00,"Авария на электролинии, остановка работы насос...",cold_water,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...,1
2,53c570099fe380dce9e56e2ace9cfa9c,2018-01-01 10:44:00,2018-01-02 18:00:00,Авария в системе водоснабжения дома. Жителям н...,hot_water,"ООО ""Управляющая компания ""Регион-ЖКХ""","Аварийная служба ООО ""Мадикс""",1
3,9c1b9ebbd9a698eef046b27cb3568745,2018-01-01 11:32:00,2018-01-01 15:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...,1
4,8aa631cb343aac0731bbde806dfa6d8c,2018-01-01 11:33:00,2018-01-01 15:00:00,"Авария на электролинии, остановка работы насос...",hot_water,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...,1


In [36]:
def get_duration_in_minutes(row: pd.Series) -> int:
    start_date = row.start_date
    end_date = row.end_date
    duration = end_date - start_date

    return duration.total_seconds() // 60

In [37]:
blackouts["duration"] = blackouts[["start_date", "end_date"]].progress_apply(get_duration_in_minutes, axis=1)

100%|██████████| 25264/25264 [00:00<00:00, 59421.78it/s]


#### Создание признака - кол-во зданий, у которых отключили систему ЖКХ

In [97]:
tmp_buildings = buildings.copy().rename(columns={
    "id": "building_id"
})[["building_id", "is_fake"]]

filtered_blackouts_buildings = blackouts_buildings.merge(tmp_buildings, on="building_id")
filtered_blackouts_buildings = filtered_blackouts_buildings[
    filtered_blackouts_buildings["is_fake"] == 0
]

blackouts_buildings_count = filtered_blackouts_buildings.groupby("blackout_id")[
    "building_id"].count().reset_index().rename(columns={
        "blackout_id": "id",
        "building_id": "buildings_count"
    })

blackouts = blackouts.merge(blackouts_buildings_count, how="left", on="id")

blackouts["buildings_count"] = blackouts["buildings_count"].fillna(0)

blackouts.head()

,id,start_date,end_date,description,type,initiator_name,source,month,duration,buildings_count
0,f88cefa506f44ebf8f010b8681b5449e,2018-01-01 00:08:00,2018-01-01 09:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...,1,532.0,95.0
1,38ddf6852801fa90cc70f9770239961e,2018-01-01 00:24:00,2018-01-01 10:00:00,"Авария на электролинии, остановка работы насос...",cold_water,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...,1,576.0,56.0
2,53c570099fe380dce9e56e2ace9cfa9c,2018-01-01 10:44:00,2018-01-02 18:00:00,Авария в системе водоснабжения дома. Жителям н...,hot_water,"ООО ""Управляющая компания ""Регион-ЖКХ""","Аварийная служба ООО ""Мадикс""",1,1876.0,1.0
3,9c1b9ebbd9a698eef046b27cb3568745,2018-01-01 11:32:00,2018-01-01 15:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...,1,208.0,27.0
4,8aa631cb343aac0731bbde806dfa6d8c,2018-01-01 11:33:00,2018-01-01 15:00:00,"Авария на электролинии, остановка работы насос...",hot_water,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...,1,207.0,4.0


Объединяем данные по отключениям с информацией о зданиях, фильтруем фейковые объекты и подсчитываем количество реальных зданий для каждого отключения. В итоге получаем таблицу отключений с дополненным признаком "количество затронутых зданий".

## Анализ данных и визуализации

In [130]:
tmp_cities = cities.copy().rename(columns={
    "id": "city_id",
    "name": "city_name"
})

tmp_streets = streets.copy().rename(columns={
    "id": "street_id",
    "name": "street_name"
})[["street_id", "street_name"]]

tmp_districts = districts.copy().rename(columns={
    "id": "district_id",
    "name": "district_name"
})

tmp_folk_districts = folk_districts.copy().rename(columns={
    "id": "folk_district_id",
    "name": "folk_district_name"
})

tmp_big_folk_districts = big_folk_districts.copy().rename(columns={
    "id": "big_folk_district_id",
    "name": "big_folk_district_name"
})

buildings_for_analysis = buildings.copy()
buildings_for_analysis = buildings_for_analysis.merge(tmp_cities, on="city_id", how="left")
buildings_for_analysis = buildings_for_analysis.merge(tmp_streets, on="street_id", how="left")
buildings_for_analysis = buildings_for_analysis.merge(tmp_districts, on="district_id", how="left")
buildings_for_analysis = buildings_for_analysis.merge(tmp_folk_districts, on="folk_district_id", how="left")
buildings_for_analysis = buildings_for_analysis.merge(tmp_big_folk_districts, on="big_folk_district_id", how="left")

buildings_for_analysis.head()

,id,street_id,number,district_id,is_fake,folk_district_id,big_folk_district_id,type,city_id,coordinates,latitude,longitude,city_name,street_name,district_name,folk_district_name,big_folk_district_name
0,b428b92bb123994a56234bb6eeeed414,37454c5a86ea320f2d5eb4eabdf1ae86,1,504f6b8bd5daeba64beb62bda8b3c12a,0,551a73c4f1b8d1cc72eea5ec752463c4,9fec0dfb8ff6bb1cf6126dc933be3489,Нежилые объекты,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.14199293270051, ""lon"": 131.9080723...",43.141993,131.908072,Владивосток,100-летия Владивостока пр-кт,Первореченский район,Днепровская,Столетие
1,24aa25303f0f91ede8ee0c0d324c8ee3,37454c5a86ea320f2d5eb4eabdf1ae86,100,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,Многоквартирные дома,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.171246518452826, ""lon"": 131.917043...",43.171247,131.917043,Владивосток,100-летия Владивостока пр-кт,Советский район,Вторая речка (Русская-Давыдова),Вторая речка
2,504f6b8bd5daeba64beb62bda8b3c12a,37454c5a86ea320f2d5eb4eabdf1ae86,100А,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,Многоквартирные дома,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17160112621735, ""lon"": 131.9178653...",43.171601,131.917865,Владивосток,100-летия Владивостока пр-кт,Советский район,Вторая речка (Русская-Давыдова),Вторая речка
3,87d459096286c57ed5e780205f041e68,37454c5a86ea320f2d5eb4eabdf1ae86,100В,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,Многоквартирные дома,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17208966911542, ""lon"": 131.9183524...",43.172090,131.918352,Владивосток,100-летия Владивостока пр-кт,Советский район,Вторая речка (Русская-Давыдова),Вторая речка
4,a4f6e19b37548845b4ee10c573809e68,37454c5a86ea320f2d5eb4eabdf1ae86,102,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,Многоквартирные дома,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.172186535721075, ""lon"": 131.917439...",43.172187,131.917440,Владивосток,100-летия Владивостока пр-кт,Советский район,Вторая речка (Русская-Давыдова),Вторая речка


Произведено объединение данных из справочников с информацией о зданиях для проведения комплексного анализа.

### Города

In [131]:
cities_count = buildings_for_analysis.groupby("city_name")["id"].count().reset_index().rename(columns={
    "city_name": "Название города",
    "id": "Кол-во зданий"
})

fig = px.bar(cities_count, x="Название города", y="Кол-во зданий", text_auto=True)
fig.update_layout(title_text="Распределение зданий по городам")
fig.show()

Анализ данных показывает наличие информации по двум городам - Владивосток и Артём, при этом объем данных по Владивостоку приблизительно вдвое превышает данные по Артёму.

### Улицы

In [132]:
streets_count = buildings_for_analysis.groupby("street_name")["id"].count().reset_index().rename(columns={
    "street_name": "Название улицы",
    "id": "Кол-во зданий"
}).sort_values(by="Кол-во зданий")[-50:]

mean_value = streets_count["Кол-во зданий"].mean()

fig = px.bar(streets_count, x="Кол-во зданий", y="Название улицы", text_auto=True, orientation="h")

fig.update_layout(
    title_text="Распределение зданий по улицам (топ-50)",
    height=1000
)

fig.show()

В топ-50 улиц по количеству зданий наблюдается относительная однородность распределения, при этом значительную часть списка занимают улицы Владивостока.

### Название района

In [133]:
district_count = buildings_for_analysis.groupby("district_name")["id"].count().reset_index().rename(columns={
    "district_name": "Название района",
    "id": "Кол-во зданий"
}).sort_values(by="Кол-во зданий")

fig = px.bar(district_count, x="Название района", y="Кол-во зданий", text_auto=True)

fig.update_layout(
    title_text="Распределение зданий по районам"
)

fig.show()

Анализ распределения зданий по официальным районам выявил значительную неравномерность, с выраженным доминированием Советского района над остальными.

### Крупные народные районы

In [134]:
big_folk_district_count = buildings_for_analysis.groupby("big_folk_district_name")["id"].count().reset_index().rename(columns={
    "big_folk_district_name": "Название крупного народного района",
    "id": "Кол-во зданий"
}).sort_values(by="Кол-во зданий")

fig = px.bar(big_folk_district_count, x="Название крупного народного района", y="Кол-во зданий", text_auto=True)
fig.update_layout(title_text="Распределение зданий по крупным народным районам")

fig.show()

Анализ распределения зданий по крупным народным районам выявил значительную неравномерность, с выраженным доминированием района Трудовой над остальными.

### Народные районы

In [135]:
folk_district_count = buildings_for_analysis.groupby("folk_district_name")["id"].count().reset_index().rename(columns={
    "folk_district_name": "Название народного района",
    "id": "Кол-во зданий"
}).sort_values(by="Кол-во зданий")

fig = px.bar(folk_district_count, x="Кол-во зданий", y="Название народного района", text_auto=True)

fig.update_layout(
    title_text="Распределение зданий по народным районам",
    height=1200
)

fig.show()

Анализ распределения зданий по народным районам выявил значительную неравномерность, с выраженным доминированием района Трудовой над остальными.

### Фейковые здания

In [136]:
buildings_fakes_count = buildings.groupby("is_fake")["id"].count().reset_index().rename(columns={
    "is_fake": "Фейковое здание",
    "id": "Кол-во зданий"
}).sort_values(by="Кол-во зданий")

fig = px.bar(buildings_fakes_count, x="Фейковое здание", y="Кол-во зданий", text_auto=True)

fig.update_layout(
    title_text="Распределение фейковых зданий"
)

fig.show()

Анализ структуры зданий на наличие фейков показало, что 12 000 зданий, представленные в датасете - фейковые.

### Типы зданий

In [137]:
buildings_types_count = buildings.groupby("type")["id"].count().reset_index().rename(columns={
    "type": "Тип здания",
    "id": "Кол-во зданий"
}).sort_values(by="Кол-во зданий")

fig = px.bar(buildings_types_count, x="Кол-во зданий", y="Тип здания", text_auto=True)

fig.update_layout(
    title_text="Распределение зданий по типам зданий",
    height=1200
)

fig.show()

Анализ структуры зданий по типам показывает значительное преобладание частных домов. При этом существенная часть данных (23.5 тысяч записей) имеет неопределенную категоризацию.

### Динамика отключений систем ЖКХ по датам

In [138]:
blackouts_dynamic = blackouts.groupby(blackouts["start_date"].dt.date)["id"].count().reset_index().rename(columns={
    "start_date": "Дата",
    "id": "Кол-во отключений"
})

fig = px.line(blackouts_dynamic, x="Дата", y="Кол-во отключений")

fig.update_layout(
    title_text="Динамика кол-ва отключений под датам"
)

fig.show()

Анализ временного ряда отключений выявил циклический паттерн: периоды высокой активности (1-2 дня) чередуются с днями значительного снижения количества инцидентов.

### Типы отключений

In [139]:
blackouts_types_count = blackouts.groupby("type")["id"].count().reset_index().rename(columns={
    "type": "Тип отключения",
    "id": "Кол-во отключений"
}).sort_values(by="Кол-во отключений")

fig = px.bar(blackouts_types_count, x="Тип отключения", y="Кол-во отключений", text_auto=True)

fig.update_layout(
    title_text="Распределение отключений по типам отключений"
)

fig.show()

Анализ распределения отключений по типам коммунальных услуг показал, что наиболее частыми являются отключения систем водоснабжения (холодного и горячего).

### Описательная статистика по отключениям систем ЖКХ

In [140]:
blackouts.describe()

,start_date,end_date,month,duration,buildings_count
count,25264,25264,25264.000000,25264.000000,25264.000000
mean,2018-12-29 18:52:18.816497408,2018-12-31 02:06:20.649936640,6.484998,1874.030557,6.806286
min,2018-01-01 00:08:00,2018-01-01 09:00:00,1.000000,1.000000,0.000000
25%,2018-07-08 09:36:00,2018-07-12 12:00:00,4.000000,172.000000,1.000000
50%,2018-12-27 16:36:00,2018-12-28 08:14:30,6.000000,281.000000,1.000000
75%,2019-06-18 11:14:00,2019-06-19 16:00:00,10.000000,645.000000,2.000000
max,2019-12-31 20:29:00,2020-01-09 12:00:00,12.000000,512593.000000,2577.000000
std,NaN,NaN,3.507516,9621.530844,38.799376


Анализ данных по отключениям за 2018-2019 годы показывает, что большинство инцидентов являются краткосрочными. Типичная продолжительность отключения составляет от 3 до 11 часов, что укладывается в рамки одного дня. Это свидетельствует о том, что большая часть аварийных ситуаций оперативно устраняется в течение рабочих суток.

### Описание в векторном пространстве

In [141]:
morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):
    s = re.sub(r'[^\w\s]+|[\d]+', r'',text).strip()
    s = s.lower()
    s1 = word_tokenize(s) #токенизация
    words=[]
    for i in s1:
        pv = morph.parse(i)
        words.append(pv[0].normal_form)
    sentence=' '.join(words)
    return sentence

In [142]:
blackouts_for_analysis = blackouts.copy()

blackouts_for_analysis["prepared_description"] = blackouts_for_analysis["description"].progress_apply(preprocess_text)

100%|██████████| 25264/25264 [00:26<00:00, 959.59it/s] 


In [143]:
russian_stopwords = stopwords.words("russian") #стоп-слова

tf_idf_vectorizer = TfidfVectorizer(max_features=500, min_df=20, max_df=0.7, stop_words=russian_stopwords)
description_tfidf = tf_idf_vectorizer.fit_transform(blackouts_for_analysis["prepared_description"])
description_tfidf_df = pd.DataFrame(description_tfidf.toarray(),columns=tf_idf_vectorizer.get_feature_names_out())
description_tfidf_df.head()

,аварийный,авария,адмирал,бойлер,бойлерная,борисенко,вентиль,вести,вестись,владивосток,...,утут,участок,фидер,холодное,центральный,чистка,электролиния,электроподстанция,электросеть,электроснабжение
0,0.0,0.424690,0.0,0.0,0.0,0.0,0.0,0.0,0.297563,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.475445
1,0.0,0.212776,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.495082,0.0,0.0,0.000000
2,0.0,0.211297,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
3,0.0,0.424690,0.0,0.0,0.0,0.0,0.0,0.0,0.297563,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.475445
4,0.0,0.212776,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.495082,0.0,0.0,0.000000


In [144]:
tsne = TSNE(n_components=2, random_state=0)

description_vectors = tsne.fit_transform(description_tfidf_df)

description_vectors[:5]

array([[-90.87036 , -62.061974],
       [-16.514227, -74.972755],
       [ 64.68096 ,  57.437687],
       [-90.87036 , -62.061974],
       [-16.514227, -74.972755]], dtype=float32)

In [145]:
duration_q1 = blackouts_for_analysis["duration"].quantile(0.25)
duration_q2 = blackouts_for_analysis["duration"].quantile(0.5)
duration_q3= blackouts_for_analysis["duration"].quantile(0.75)

In [146]:
def get_blackout_duration_group(duration: float):
    if duration >= duration_q3:
        return "Продолжительные"
    elif duration >= duration_q2:
        return "Средней продолжительности" 
    elif duration >= duration_q1:
        return "Кратковременные"
    else:
        return "Минимальные"

In [147]:
blackouts_for_analysis["duration_group"] = blackouts_for_analysis["duration"].progress_apply(
    get_blackout_duration_group
)

100%|██████████| 25264/25264 [00:00<00:00, 1130087.30it/s]


In [148]:
description_for_analysis = pd.DataFrame(description_vectors, columns=["Признак 1", "Признак 2"])
description_for_analysis["Тип отключения"] = blackouts_for_analysis["type"]
description_for_analysis["Продолжительность отключения"] = blackouts_for_analysis["duration_group"]

Мы взяли описания отключений и сначала почистили текст — убрали знаки препинания, привели слова к нормальной форме через лемматизацию. Потом перевели слова в числа с помощью TF-IDF, чтобы оценить важность каждого слова в контексте всех описаний.

Дальше было интересно посмотреть, как эти описания группируются по смыслу. Для этого использовали t-SNE — алгоритм, который хорошо умеет визуализировать подобные вещи, сжимая многомерные данные до 2D.

Мы выбрали такие методы, потому что они хорошо подходят для коротких технических текстов: TF-IDF подчеркивает ключевые термины, а t-SNE отлично показывает скрытые структуры в данных (и оставляет нелинейные зависимости, в отличие от PCA). Всё вместе подтвердило, что даже по коротким описаниям можно автоматически различать типы проблем в ЖКХ.

In [149]:
fig = px.scatter(description_for_analysis, x="Признак 1", y="Признак 2", color="Тип отключения")

fig.update_layout(
    title_text="Распределение типов отключений относительно описания отключения",
    height=1000,
    width=1000
)

fig.show()

На графике чётко видны три основные группы: электричество, водоснабжение (холодная и горячая вода рядом, что логично — тематика похожая) и отопление.

In [178]:
fig = px.scatter(description_for_analysis, x="Признак 1", y="Признак 2", color="Продолжительность отключения")

fig.update_layout(
    title_text="Распределение типов отключений относительно продолжительности отключения",
    height=1000,
    width=1000
)

fig.show()

На графике явно выделяются две основные категории: продолжительные отключения (свыше 75-го перцентиля) и минимальные по длительности (ниже 25-го перцентиля).

In [151]:
blackouts_for_analysis["initiator_name"].nunique()

369

### Инициатор отключения в векторном пространстве

In [152]:
blackouts_for_analysis["prepared_initiator_name"] = blackouts_for_analysis["initiator_name"].progress_apply(preprocess_text)

100%|██████████| 25264/25264 [00:20<00:00, 1228.59it/s]


In [153]:
russian_stopwords = stopwords.words("russian") #стоп-слова

tf_idf_vectorizer = TfidfVectorizer(max_features=500, min_df=20, max_df=0.7, stop_words=russian_stopwords)
initiator_name_tfidf = tf_idf_vectorizer.fit_transform(blackouts_for_analysis["prepared_initiator_name"])
initiator_name_tfidf_df = pd.DataFrame(initiator_name_tfidf.toarray(),columns=tf_idf_vectorizer.get_feature_names_out())
initiator_name_tfidf_df.head()

,аварийный,альтакса,альянс,ао,арсо,атлант,багратион,бриз,варяг,викс,...,хороший,центр,центральный,цжка,чуркин,шароват,эгершёльд,электрический,электросеть,эра
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.424811,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.424811,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400764,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400764,0.0,0.0


In [154]:
tsne = TSNE(n_components=2, random_state=0)

initiator_name_vectors = tsne.fit_transform(initiator_name_tfidf_df)

initiator_name_vectors[:5]

array([[157.59007 ,  49.653755],
       [157.59007 ,  49.653755],
       [-75.87362 , 208.956   ],
       [111.80448 ,  20.391087],
       [111.80448 ,  20.391087]], dtype=float32)

In [155]:
initiator_name_for_analysis = pd.DataFrame(initiator_name_vectors, columns=["Признак 1", "Признак 2"])
initiator_name_for_analysis["Тип отключения"] = blackouts_for_analysis["type"]
initiator_name_for_analysis["Продолжительность отключения"] = blackouts_for_analysis["duration_group"]

In [179]:
fig = px.scatter(initiator_name_for_analysis, x="Признак 1", y="Признак 2", color="Тип отключения")

fig.update_layout(
    title_text="Распределение типов отключений относительно описания отключения",
    height=1000,
    width=1000
)

fig.show()

Не графике не видно чётких различий между инициатором и типом отключения.

In [180]:
fig = px.scatter(initiator_name_for_analysis, x="Признак 1", y="Признак 2", color="Продолжительность отключения")

fig.update_layout(
    title_text="Распределение типов отключений относительно продолжительности отключения",
    height=1000,
    width=1000
)

fig.show()

На графике не видно чётких различий между инициатором и продолжительностью отключения.

### Средняя продолжительность отключения по типам отключений

In [158]:
blackout_type_mean_duration = blackouts_for_analysis.groupby("type")["duration"].mean().reset_index().rename(
    columns={
        "type": "Тип отключения",
        "duration": "Средняя продолжительность (в мин.)"
    }
).sort_values("Средняя продолжительность (в мин.)")

fig = px.bar(blackout_type_mean_duration, x="Тип отключения", y="Средняя продолжительность (в мин.)", text_auto=True)

fig.update_layout(
    title_text="Средняя продолжительность отключений по типам отключений"
)
fig.update_traces(texttemplate='%{y:.1f}')

fig.show()

Быстрее всего устраняют проблемы с электричеством — в среднем за 4,5 часа. Отключения холодной воды и отопления длятся около 10 часов, а горячую воду восстанавливают дольше всего — примерно 3 суток.

### Средняя продолжительность отключения по инициаторам (топ-50)

In [159]:
blackout_initiator_mean_duration = blackouts_for_analysis.groupby("initiator_name")["duration"].mean().reset_index().rename(
    columns={
        "initiator_name": "Инициатор",
        "duration": "Средняя продолжительность (в мин.)"
    }
).sort_values("Средняя продолжительность (в мин.)")[-50:]

blackout_initiator_mean_duration["Средняя продолжительность (в мин.)"] = blackout_initiator_mean_duration["Средняя продолжительность (в мин.)"].round().astype(int)

fig = px.bar(blackout_initiator_mean_duration, x="Средняя продолжительность (в мин.)", y="Инициатор", text_auto=True)

fig.update_layout(
    title_text="Средняя продолжительность отключений по инициаторам (топ-50)",
    height=1000
)


fig.show()

Анализ графика показывает зависимость продолжительности отключения от организации-инициатора.

### Средняя продолжительность отключения по месяцам

In [160]:
month_mean_duration = blackouts_for_analysis.groupby("month")["duration"].mean().reset_index().rename(
    columns={
        "month": "Месяц",
        "duration": "Средняя продолжительность (в мин.)"
    }
)

month_mean_duration["Средняя продолжительность (в мин.)"] = month_mean_duration["Средняя продолжительность (в мин.)"].round().astype(int)

fig = px.bar(month_mean_duration, x="Месяц", y="Средняя продолжительность (в мин.)", text_auto=True)

fig.update_layout(
    title_text="Средняя продолжительность отключений по месяцам"
)
fig.update_xaxes(
    tickmode='linear',
    dtick=1
)

fig.show()

Анализ графика показал, что средняя продолжительность отключений больше в тёплые месяцы года (май, июнь, июль, август, сентябрь, октябрь).

### Корреляция между продолжительностью отключения и кол-вом "затрагиваемых" зданий

In [169]:
pearson_corr = blackouts_for_analysis[["duration", "buildings_count"]].corr()["buildings_count"].duration
spearman_corr = blackouts_for_analysis[["duration", "buildings_count"]].corr("spearman")["buildings_count"].duration
kendall_corr = blackouts_for_analysis[["duration", "buildings_count"]].corr("kendall")["buildings_count"].duration

print("Корреляция Пирсона:", pearson_corr)
print("Корреляция Спирмена:", spearman_corr)
print("Корреляция Кендалла:", kendall_corr)

Корреляция Пирсона: 0.02911469162980069
Корреляция Спирмена: 0.14963671062404527
Корреляция Кендалла: 0.11448014099291855


In [177]:
fig = px.scatter(blackouts_for_analysis, x="buildings_count", y="duration")

fig.update_layout(
    title_text="Зависимость продолжительности отключения от кол-ва зданий",
    yaxis={"title": "Продолжительность отключения (в мин.)"},
    xaxis={"title": "Кол-во затрагиваемых зданий"}
)

fig.show()

Корреляционный анализ не выявил зависимости между кол-вом затрагиваемых зданий и продолжительностью отключения (в мин.)